In [2]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.utils import generate_mask_tensor
from src.embedding import embed
from src.gp_ccm import GP_ccm_sig
from src.sp_ccm import run_SP_CCM, SP_CCM_iaaft, run_ccm_experiment
from src.iaaft import surrogates

from scipy.stats import ranksums
torch.set_printoptions(sci_mode = False)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# Data generation

In [4]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
c = torch.tensor([0.2])
a = torch.tensor([0.3])
b = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    c_next = c[t] * (3.9 - (3.9 * c[t]))
    a_next = a[t] * (3.8 - (3.8 * a[t]) - (0.3 * c[t])) # previously 0.2
    b_next = b[t] * (3.6 - (3.6 * b[t]) - (0.2 * c[t]))

    c = torch.concat((c, c_next.unsqueeze(0)))
    a = torch.concat((a, a_next.unsqueeze(0)))
    b = torch.concat((b, b_next.unsqueeze(0)))
            
# Normalising step
c_norm = c.sub(c.mean(dim = -1).unsqueeze(-1)).div(c.std(dim = -1).unsqueeze(-1))
a_norm = a.sub(a.mean(dim = -1).unsqueeze(-1)).div(a.std(dim = -1).unsqueeze(-1))
b_norm = b.sub(b.mean(dim = -1).unsqueeze(-1)).div(b.std(dim = -1).unsqueeze(-1))

In [ ]:
# Try running on diff signature instead
c_norm.diff()

tensor([ 1.4591,  1.0015, -2.1054,  1.7921, -0.8889,  1.3508, -2.7596,  1.2317,
         1.5831, -3.0154,  0.8488,  1.8453, -1.6228,  1.8363, -2.5224,  1.5035,
         0.8399, -1.7615,  1.8545, -2.1479,  1.7735, -0.7120,  1.1424, -2.3902,
         1.6208,  0.3143, -0.6283,  1.0324, -2.1695,  1.7631, -0.6209,  1.0224,
        -2.1487,  1.7731, -0.7085,  1.1379, -2.3815,  1.6277,  0.2780, -0.5529,
         0.9275, -1.9494,  1.8396, -1.5071,  1.8027, -2.7598,  1.2315,  1.5835,
        -3.0156,  0.8485,  1.8452, -1.6206,  1.8358, -2.5275,  1.4985,  0.8591,
        -1.8029,  1.8551, -2.0187,  1.8225, -1.2402,  1.6624, -3.0237,  0.8349,
         1.8405, -1.5231,  1.8083, -2.7313,  1.2683,  1.5140, -2.9666,  0.9291,
         1.8545, -2.1489,  1.7730, -0.7075,  1.1366, -2.3789,  1.6297,  0.2673,
        -0.5308,  0.8955, -1.8812,  1.8503, -1.7552,  1.8542, -2.1668,  1.7644,
        -0.6322,  1.0377, -2.1804,  1.7577, -0.5749,  0.9586, -2.0155,  1.8234,
        -1.2528,  1.6709, -3.0204,  0.84

In [ ]:
def affirm_uniqueness(ts):
    # Check for duplicates and add noise 
    dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
    print("There are ", dupes, " duplicates in the timeseries.")
    if dupes > 0:
        noise_level = 0.001
        while dupes > 0:
            ts = ts + torch.randn(ts.shape[0]) * noise_level
            # recalculate dupes
            dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
            print("Now we have ", dupes, " dupes.")
    return ts

# affirm_uniqueness(c_norm)

# Visualise

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = c[0:41],
                    mode = 'lines',
                    name = 'C',
                    line_color = "forestgreen"))

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = a[0:41],
                    mode = 'lines',
                    name = 'A',
                    line_color = "blue"))

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = b[0:41],
                    mode = 'lines',
                    name = 'B',
                    line_color = "cornflowerblue"))

fig.update_layout(title = 'Confounding time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2,41])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

In [5]:
# define k globally
k = 3

##############
### GP-CCM ###
##############
sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) # k -1 

# too low noise breaks it
noise_scalar = torch.tensor([0.05], device = device)

############
### ECCM ###
############
ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device)

N_TRAIN = torch.tensor([100]).to(device)

# C -> A

In [ ]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = a_norm.to(device), # Testing X -> Y
                           x = c_norm.to(device),
                           max_pos_offset = cp_shift, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)
# break_points = torch.randperm(n = x_gt.shape[0]) for swap model

rho_l = torch.empty(size = (1, 0)).to(device)
rho_l_null = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
        
        # Null model
        # cx_gt_swap = torch.cat((x_gt[break_points[l]:], x_gt[:break_points[l]]))

        rho_null, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                # x_train = x_gt_swap[l_train_masks[l]].to(device),
                # x_test = x_gt_swap[~ l_train_masks[l]].to(device),
                x_train = x_gt_iaaft[l, l_train_masks[l]].to(device),
                x_test = x_gt_iaaft[l, ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
        rho_l_null = torch.concat((rho_l_null, rho_null.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho nullmodel mean:", rho_l_null.mean().item())

ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_null, quants)[1].item())

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3216.22it/s]


rho mean: 0.720628559589386
rho nullmodel mean: 0.005061199888586998
rho null upper (p95) 0.10159317404031754


In [6]:
CA_ccm_rho_mean, CA_ccm_rho_sd, CA_ccm_rho_ind_p95, CA_ccm_noise =  run_ccm_experiment(
    causal_x = c_norm.to(device),
    causal_y = a_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
# One fast pass
rho_l = run_SP_CCM(y = a_norm.to(device),
                   x = c_norm.to(device),
                   filter = ccm_filter.to(device),
                   max_offset = torch.tensor(ccm_shift).to(device),
                   L = 100,
                   device = device)

# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = a_norm.cpu(), ns = a_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(a_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(y = a_norm.to(device), 
                       # y = (a_norm + torch.randn(c_norm.shape[0]) * 0.01).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l], 
                       selected_train_mask = l_train_masks[l], 
                       max_offset = ccm_shift,
                       ccmfilter = ccm_filter, 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())

/tmp/ipykernel_1245094/3362176959.py:5: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 4185.38it/s]


rho mean: 0.8189247250556946
rho ind mean: 0.012063790112733841
rho ind upper (p95) 0.12498953938484192


# C -> B

In [ ]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = b_norm.to(device), # Testing X -> Y
                           x = c_norm.to(device),
                           max_pos_offset = cp_shift, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)
# break_points = torch.randperm(n = x_gt.shape[0]) for swap model

rho_l = torch.empty(size = (1, 0)).to(device)
rho_l_null = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
        
        # Null model
        # cx_gt_swap = torch.cat((x_gt[break_points[l]:], x_gt[:break_points[l]]))

        rho_null, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                # x_train = x_gt_swap[l_train_masks[l]].to(device),
                # x_test = x_gt_swap[~ l_train_masks[l]].to(device),
                x_train = x_gt_iaaft[l, l_train_masks[l]].to(device),
                x_test = x_gt_iaaft[l, ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
        rho_l_null = torch.concat((rho_l_null, rho_null.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho nullmodel mean:", rho_l_null.mean().item())

ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_null, quants)[1].item())

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3339.85it/s]


rho mean: 0.7021216154098511
rho nullmodel mean: 0.0018535393755882978
rho null upper (p95) 0.10637050867080688


In [ ]:
# One fast pass
rho_l = run_SP_CCM(y = b_norm.to(device),
                   x = c_norm.to(device),
                   filter = ccm_filter.to(device),
                   max_offset = torch.tensor(ccm_shift).to(device),
                   L = 100,
                   device = device)

# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = c_norm.cpu(), ns = a_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(c_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(y = b_norm.to(device), 
                       # y = (a_norm + torch.randn(c_norm.shape[0]) * 0.01).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l], 
                       selected_train_mask = l_train_masks[l], 
                       max_offset = ccm_shift,
                       ccmfilter = ccm_filter, 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())

/tmp/ipykernel_1245094/1169111137.py:5: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 4242.81it/s]


rho mean: 0.8747581243515015
rho ind mean: 0.01083726342767477
rho ind upper (p95) 0.13111068308353424


# A -> B

In [ ]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = b_norm.to(device), # Testing X -> Y
                           x = a_norm.to(device),
                           max_pos_offset = cp_shift, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)
# break_points = torch.randperm(n = x_gt.shape[0]) for swap model

rho_l = torch.empty(size = (1, 0)).to(device)
rho_l_null = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
        
        # Null model
        # cx_gt_swap = torch.cat((x_gt[break_points[l]:], x_gt[:break_points[l]]))

        rho_null, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                # x_train = x_gt_swap[l_train_masks[l]].to(device),
                # x_test = x_gt_swap[~ l_train_masks[l]].to(device),
                x_train = x_gt_iaaft[l, l_train_masks[l]].to(device),
                x_test = x_gt_iaaft[l, ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
        rho_l_null = torch.concat((rho_l_null, rho_null.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho nullmodel mean:", rho_l_null.mean().item())

ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_null, quants)[1].item())

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3319.11it/s]


rho mean: 0.07906999439001083
rho nullmodel mean: 0.0017633360112085938
rho null upper (p95) 0.10077075660228729


In [ ]:
# One fast pass
rho_l = run_SP_CCM(y = b_norm.to(device),
                   x = a_norm.to(device),
                   filter = ccm_filter.to(device),
                   max_offset = torch.tensor(ccm_shift).to(device),
                   L = 100,
                   device = device)

# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = a_norm.cpu(), ns = a_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(a_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(y = b_norm.to(device), 
                       # y = (b_norm + torch.randn(c_norm.shape[0]) * 0.01).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l], 
                       selected_train_mask = l_train_masks[l], 
                       max_offset = ccm_shift,
                       ccmfilter = ccm_filter, 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())

/tmp/ipykernel_1245094/1705970036.py:5: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 4176.22it/s]


rho mean: 0.10485821217298508
rho ind mean: 0.006852763704955578
rho ind upper (p95) 0.11363905668258667


# A -> C

In [ ]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = c_norm.to(device), # Testing X -> Y
                           x = a_norm.to(device),
                           max_pos_offset = cp_shift, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)
# break_points = torch.randperm(n = x_gt.shape[0]) for swap model

rho_l = torch.empty(size = (1, 0)).to(device)
rho_l_null = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
        
        # Null model
        # cx_gt_swap = torch.cat((x_gt[break_points[l]:], x_gt[:break_points[l]]))

        rho_null, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                # x_train = x_gt_swap[l_train_masks[l]].to(device),
                # x_test = x_gt_swap[~ l_train_masks[l]].to(device),
                x_train = x_gt_iaaft[l, l_train_masks[l]].to(device),
                x_test = x_gt_iaaft[l, ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
        rho_l_null = torch.concat((rho_l_null, rho_null.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho nullmodel mean:", rho_l_null.mean().item())

ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_null, quants)[1].item())

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3131.42it/s]


rho mean: -0.017027076333761215
rho nullmodel mean: 0.004680793732404709
rho null upper (p95) 0.10873879492282867


In [ ]:
# One fast pass
rho_l = run_SP_CCM(y = (c_norm + torch.randn(c_norm.shape[0]) * 0.2).to(device),
                   x = a_norm.to(device),
                   filter = ccm_filter.to(device),
                   max_offset = torch.tensor(ccm_shift).to(device),
                   L = 100,
                   device = device)

# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = a_norm.cpu(), ns = a_norm.shape[0], tol_pc = 10, verbose = False), dtype = torch.float32)

l_train_masks = generate_mask_tensor(a_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(# y = c_norm.to(device), 
                       y = (c_norm + torch.randn(c_norm.shape[0]) * 0.2).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l], 
                       selected_train_mask = l_train_masks[l], 
                       max_offset = ccm_shift,
                       ccmfilter = ccm_filter, 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())

/tmp/ipykernel_1245094/2927874530.py:5: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



rho mean: 0.013830607756972313
rho ind mean: 0.005405516363680363
rho ind upper (p95) 0.10783148556947708


In [ ]:
def run_ccm_experiment(causal_x, causal_y, ccm_filter, ccm_shift, n_train, device):
    
    result_status = True

    noise_increment = torch.tensor([0.025], device = device)
    noise_creeper = torch.tensor([0.], device = device) - noise_increment

    # True: we have nan
    while (result_status == True):

        # increment noise (0 in first iteration)
        noise_creeper = noise_creeper + noise_increment

        # Add noise (0 in first iteration)
        causal_y = causal_y + (torch.randn(causal_y.shape[0], device = device) * noise_creeper)
        
        ###########
        ### CCM ###
        ###########

        rho_l = run_SP_CCM(
                        y = causal_y,
                        x = causal_x,
                        filter = ccm_filter,
                        max_offset = ccm_shift,
                        L = n_train,
                        device = device)
        
        #######################
        ### CCM INDEPENDENT ###
        #######################

        # Generate N surrogate (permuted ts)
        causal_y_N_iaaft = torch.tensor(surrogates(x = causal_y.cpu(), ns = causal_y.shape[0], tol_pc = 5., verbose = False), dtype = torch.float32)

        l_train_masks = generate_mask_tensor(a_norm.shape[0], n_train)

        # placeholder
        rho_l_ind = torch.empty(size = (1, 0)).to(device)

        # N passes with a different surrogate each time
        for l in range(causal_y_N_iaaft.shape[0]):

            rho_ind = SP_CCM_iaaft(y = causal_y, 
                            x_gt_iaaft_selected = x_gt_iaaft[l], 
                            selected_train_mask = l_train_masks[l], 
                            max_offset = ccm_shift,
                            ccmfilter = ccm_filter, 
                            device = device)
            
            rho_l_ind = torch.cat((rho_l_ind, rho_ind.unsqueeze(0).unsqueeze(0)), dim = 1)

        p95 = torch.tensor([0.95]).to(device)

        result_status = (rho_l.isnan().any() & rho_l_ind.isnan().any())

        if (result_status == True):
            print("We get nan's and have to increase the noise level.")

    # print once while loop is finished
    print("Rho mean", np.round(rho_l.mean().item(), 3))
    print("Rho std", np.round(rho_l.std().item(), 3))
    print("Rho indep. p95", np.round(torch.quantile(rho_l_ind, p95).item(), 3))

    print("Added noise", np.round(noise_creeper.item(), 3))

    return(rho_l.mean(), rho_l.std(), torch.quantile(rho_l_ind, p95), noise_creeper)

In [ ]:
AC_ccm_rho_mean, AC_ccm_rho_sd, AC_ccm_rho_ind_p95, AC_ccm_noise =  run_ccm_experiment(
    causal_x = a_norm.to(device),
    causal_y = c_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

We get nan's and have to increase the noise level.
Rho mean nan
Rho std nan
Rho indep. p95 nan
Added noise 0.0
Rho mean 0.052
Rho std 0.066
Rho indep. p95 0.113
Added noise 0.02500000037252903


# B -> A

In [ ]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = a_norm.to(device), # Testing X -> Y
                           x = b_norm.to(device),
                           max_pos_offset = cp_shift, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)
# break_points = torch.randperm(n = x_gt.shape[0]) for swap model

rho_l = torch.empty(size = (1, 0)).to(device)
rho_l_null = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
        
        # Null model
        # cx_gt_swap = torch.cat((x_gt[break_points[l]:], x_gt[:break_points[l]]))

        rho_null, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                # x_train = x_gt_swap[l_train_masks[l]].to(device),
                # x_test = x_gt_swap[~ l_train_masks[l]].to(device),
                x_train = x_gt_iaaft[l, l_train_masks[l]].to(device),
                x_test = x_gt_iaaft[l, ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
        rho_l_null = torch.concat((rho_l_null, rho_null.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho nullmodel mean:", rho_l_null.mean().item())

ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_null, quants)[1].item())

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3342.52it/s]


rho mean: 0.015219415538012981
rho nullmodel mean: 0.007191984914243221
rho null upper (p95) 0.11314723640680313


In [ ]:
# One fast pass
rho_l = run_SP_CCM(y = a_norm.to(device),
                   x = b_norm.to(device),
                   filter = ccm_filter.to(device),
                   max_offset = torch.tensor(ccm_shift).to(device),
                   L = 100,
                   device = device)

# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = b_norm.cpu(), ns = a_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(a_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(y = a_norm.to(device), 
                       # y = (c_norm + torch.randn(c_norm.shape[0]) * 0.01).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l], 
                       selected_train_mask = l_train_masks[l], 
                       max_offset = ccm_shift,
                       ccmfilter = ccm_filter, 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())

/tmp/ipykernel_1245094/1289251318.py:5: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 4115.73it/s]


rho mean: 0.11358258873224258
rho ind mean: 0.013990476727485657
rho ind upper (p95) 0.15829266607761383


# B -> C

In [ ]:
y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = c_norm.to(device), # Testing X -> Y
                           x = b_norm.to(device),
                           max_pos_offset = cp_shift, 
                           device = device)

N = y_embeddings.shape[0]
E = y_embeddings.shape[1]

l_train_masks = generate_mask_tensor(N, 100)

x_gt_iaaft = torch.tensor(surrogates(x = x_gt.cpu(), ns = N, tol_pc = 10), dtype = torch.float32)
# break_points = torch.randperm(n = x_gt.shape[0]) for swap model

rho_l = torch.empty(size = (1, 0)).to(device)
rho_l_null = torch.empty(size = (1, 0)).to(device)

for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
        
        # Null model
        # cx_gt_swap = torch.cat((x_gt[break_points[l]:], x_gt[:break_points[l]]))

        rho_null, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                # x_train = x_gt_swap[l_train_masks[l]].to(device),
                # x_test = x_gt_swap[~ l_train_masks[l]].to(device),
                x_train = x_gt_iaaft[l, l_train_masks[l]].to(device),
                x_test = x_gt_iaaft[l, ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.3,
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
        rho_l_null = torch.concat((rho_l_null, rho_null.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho nullmodel mean:", rho_l_null.mean().item())

ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho null upper (p95)", torch.quantile(rho_l_null, quants)[1].item())

Estim100%|██████████████████████████████| 398/398 [00:00<00:00, 3373.39it/s]


rho mean: -0.03127139061689377
rho nullmodel mean: -0.0033282784279435873
rho null upper (p95) 0.11254154145717621


In [ ]:
# One fast pass
rho_l = run_SP_CCM(y = (c_norm + torch.randn(c_norm.shape[0]) * 0.05).to(device),
                   x = b_norm.to(device),
                   filter = ccm_filter.to(device),
                   max_offset = torch.tensor(ccm_shift).to(device),
                   L = 100,
                   device = device)

# N different aurrogates
x_gt_iaaft = torch.tensor(surrogates(x = b_norm.cpu(), ns = a_norm.shape[0], tol_pc = 10), dtype = torch.float32)

l_train_masks = generate_mask_tensor(a_norm.shape[0], 100)

rho_l_ind = torch.empty(size = (1, 0)).to(device)
for l in range(x_gt_iaaft.shape[0]):
    rho = SP_CCM_iaaft(y = (c_norm + torch.randn(c_norm.shape[0]) * 0.1).to(device), # add noise to embedding
                       # y = (c_norm + torch.randn(c_norm.shape[0]) * 0.01).to(device), 
                       x_gt_iaaft_selected = x_gt_iaaft[l], 
                       selected_train_mask = l_train_masks[l], 
                       max_offset = ccm_shift,
                       ccmfilter = ccm_filter, 
                       device = device)
    
    rho_l_ind = torch.cat((rho_l_ind, rho.unsqueeze(0).unsqueeze(0)), dim = 1)

print("rho mean:", rho_l.mean().item())
print("rho ind mean:", rho_l_ind.mean().item())

# ranksums(rho_l.cpu().squeeze(), rho_l_null.cpu().squeeze())

quants = torch.tensor([0.05, 0.95]).to(device)
print("rho ind upper (p95)", torch.quantile(rho_l_ind, quants)[1].item())

/tmp/ipykernel_1245094/442797924.py:5: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).

Estim100%|██████████████████████████████| 400/400 [00:00<00:00, 3924.88it/s]


rho mean: -0.0697118490934372
rho ind mean: -0.008729896508157253
rho ind upper (p95) 0.11261086165904999


# X -> Y

In [ ]:
shifts = torch.arange(-4, 4 + 1, 1)
print(shifts)

# k = 2
# fix filter
# sig_filter = torch.ones(size = (k, )).to(device)
# noise_scalar = torch.tensor([0.10], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = y_norm.to(device), # Testing X -> Y
                           x = x_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()

fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for X -> Y',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

tensor([-4, -3, -2, -1,  0,  1,  2,  3,  4])
-4


NameError: name 'y_norm' is not defined

# Y -> X

In [ ]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# k = 2
# fix filter
# sig_filter = torch.ones(size = (k, )).to(device)
# noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Y -> X
                           x = y_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Y -> X',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

# Z -> X

In [ ]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# k = 2
# fix filter
# sig_filter = torch.ones(size = (k, )).to(device)
# noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = x_norm.to(device), # Testing Z -> X
                           x = z_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Z -> X',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

# Z -> Y

In [ ]:
shifts = torch.arange(-4, 4 + 1, 1)
print(shifts)

# k = 2
# fix filter
# sig_filter = torch.ones(size = (k, )).to(device)
# noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = y_norm.to(device), # Testing Z -> Y
                           x = z_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Z -> Y',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

# X -> Z

In [ ]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# k = 2
# fix filter
# sig_filter = torch.ones(size = (k, )).to(device)
# noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = z_norm.to(device), # Testing Z -> X
                           x = x_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Z -> X',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()

# Y -> Z

In [ ]:
shifts = torch.arange(- 4, 4 + 1, 1)
print(shifts)

# k = 2
# fix filter
# sig_filter = torch.ones(size = (k, )).to(device)
# noise_scalar = torch.tensor([0.05], device = device)

rho_s = torch.zeros(size = (shifts.shape[0], 2))

for i, s in enumerate(shifts):
    print(s.item())

    y_embeddings, x_gt = embed(filter = sig_filter, 
                           y = z_norm.to(device), # Testing Y -> Z 
                           x = y_norm.to(device),
                           max_pos_offset = s, 
                           device = device)
    
    # N now changes slightly
    N = y_embeddings.shape[0]
    E = y_embeddings.shape[1]

    l_train_masks = generate_mask_tensor(N, 100)

    rho_l = torch.empty(size = (1, 0)).to(device)

    for l in range(N):   
        rho, nlml = GP_ccm_sig(
                y_embeddings_train = y_embeddings[l_train_masks[l]].unsqueeze(-1).to(device),
                y_embeddings_test = y_embeddings[ ~ l_train_masks[l]].unsqueeze(-1).to(device),
                x_train = x_gt[l_train_masks[l]].to(device),
                x_test = x_gt[ ~ l_train_masks[l]].to(device),
                noise = noise_scalar,
                rbf_sigma = 0.2, # 0.2 is good
                device = device)
                
        rho_l = torch.concat((rho_l, rho.unsqueeze(0).unsqueeze(0)), dim = 1)
     
    rho_s[i, 0] = rho_l.mean()
    rho_s[i, 1] = rho_l.std()


fig = go.Figure()

fig.add_trace(go.Scatter(x = -shifts, y = rho_s[:, 0], # reversing the meaning of x
                    mode = 'lines+markers',
                    name = 'mean',
                    line_color = "#C00000"))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] + rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines'
))

fig.add_trace(go.Scatter(
    name = "",
    x = - shifts,
    y = rho_s[:, 0] - rho_s[:, 1].mul(1),
    marker = dict(color = 'rgba(243, 176, 210, 0.8)'),
    showlegend = False,
    mode = 'lines',
    fillcolor = 'rgba(243, 176, 210, 0.3)',
    fill = 'tonexty'
))

fig.add_vline(x = 0.)
fig.add_vline(x = -(k-1), line_dash = "dash")

fig.update_layout(title = 'Cross-mapping skill for Y -> Z',
                   xaxis_title = 'shift',
                   yaxis_title = 'rho')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")

fig.show()